In [7]:
# install cell
!pip install yfinance
!pip install alpaca-py
!pip install plotly

In [8]:
#imports
import yfinance as yf
import pandas as pd
import numpy as np

In [9]:
# connection code
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

API_KEY    = "key"
API_SECRET = "secret"

client = TradingClient(API_KEY, API_SECRET, paper=True)
account = client.get_account()
print(f"Status:       {account.status}")
print(f"Buying power: ${float(account.buying_power):,.2f}")

Status:       AccountStatus.ACTIVE
Buying power: $283,076.30


In [10]:
# universe & data
tickers = ['SPY', 'QQQ', 'IWM',   # US equity
           'GLD', 'SLV',           # commodities
           'TLT', 'IEF',           # bonds
           'XLE', 'XLK', 'XLV']   # sectors

data = yf.download(tickers, start='2020-01-01')['Close']

/tmp/ipykernel_6566/2935818204.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start='2020-01-01')['Close']
[*********************100%***********************]  10 of 10 completed


In [11]:
# trend signal
# Rule: if price > 200-day moving average → hold it
#       if price < 200-day moving average → go to cash

sma200 = data.rolling(200).mean()

# Signal: 1 = hold, 0 = cash
signal = (data > sma200).astype(int)

# Equal weight among assets that are above their 200d MA
weights = signal.div(signal.sum(axis=1), axis=0).fillna(0)

print("Latest signals:")
print(signal.iloc[-1][signal.iloc[-1] == 1])

Latest signals:
Ticker
IWM    1
QQQ    1
SLV    1
SPY    1
XLE    1
XLK    1
XLV    1
Name: 2026-06-17 00:00:00, dtype: int64


In [12]:
# backtest
daily_returns = data.pct_change()
port_returns  = (weights.shift(1) * daily_returns).sum(axis=1)
benchmark     = daily_returns['SPY']

cumulative = (1 + port_returns).cumprod()
buy_hold   = (1 + benchmark).cumprod()

def sharpe(returns, rf=0.02):
    excess = returns - rf/252
    return (excess.mean() / excess.std()) * np.sqrt(252)

def max_drawdown(returns):
    cum   = (1 + returns).cumprod()
    peak  = cum.cummax()
    return ((cum - peak) / peak).min()

print(f"Strategy:     {cumulative.iloc[-1]-1:.1%}")
print(f"Buy & Hold:   {buy_hold.iloc[-1]-1:.1%}")
print(f"Sharpe:       {sharpe(port_returns):.2f}")
print(f"Max Drawdown: {max_drawdown(port_returns):.1%}")

Strategy:     119.9%
Buy & Hold:   152.8%
Sharpe:       0.76
Max Drawdown: -23.7%


In [13]:
# holding cell
latest = weights.iloc[-1]
picks  = latest[latest > 0]

print("=== CURRENT HOLDINGS ===")
if len(picks) == 0:
    print("All assets below 200d MA — hold CASH")
else:
    for ticker, w in picks.items():
        print(f"  {ticker}: {w:.0%}")

=== CURRENT HOLDINGS ===
  IWM: 14%
  QQQ: 14%
  SLV: 14%
  SPY: 14%
  XLE: 14%
  XLK: 14%
  XLV: 14%


In [14]:
# 200d MA Rankings
import plotly.graph_objects as go

# Calculate how far each stock is from its 200d MA
distance    = ((data - sma200) / sma200 * 100).iloc[-1]
latest_sig  = signal.iloc[-1]

# Build summary table
summary = pd.DataFrame({
    'Price':       data.iloc[-1],
    'MA200':       sma200.iloc[-1],
    'Distance %':  distance,
    'Signal':      latest_sig
}).dropna()

summary['Status'] = summary['Signal'].map(
    {1: 'ABOVE ▲', 0: 'BELOW ▼'}
)
summary = summary.sort_values(
    'Distance %', ascending=False
)

# Print full ranked table
above = summary[summary['Signal'] == 1]
below = summary[summary['Signal'] == 0]

print("=" * 65)
print(f"  200-DAY MA RANKINGS — "
      f"{data.index[-1].strftime('%B %d, %Y')}")
print("=" * 65)
print(f"\n  {len(above)} stocks ABOVE 200d MA  |  "
      f"{len(below)} stocks BELOW 200d MA\n")

print(f"  {'Rank':<6} {'Ticker':<8} {'Price':>8} "
      f"{'200d MA':>8} {'Distance':>10} {'Status':<12}")
print(f"  {'-'*6} {'-'*8} {'-'*8} "
      f"{'-'*8} {'-'*10} {'-'*12}")

for rank, (ticker, row) in enumerate(
    summary.iterrows(), 1
):
    marker = ' ◄ HOLDING' if rank <= 20 \
             and row['Signal'] == 1 else ''
    print(f"  {rank:<6} {ticker:<8} "
          f"${row['Price']:>7.2f} "
          f"${row['MA200']:>7.2f} "
          f"{row['Distance %']:>+9.1f}% "
          f"{row['Status']:<12}"
          f"{marker}")

# Chart — top 30 above and bottom 30 below
top30    = summary.head(30)
bottom30 = summary.tail(30)
combined = pd.concat([bottom30, top30])

colors = [
    '#1D9E75' if s == 1 else '#D85A30'
    for s in combined['Signal']
]

fig = go.Figure(go.Bar(
    x=combined.index,
    y=combined['Distance %'],
    marker_color=colors,
    text=[f"{v:+.1f}%"
          for v in combined['Distance %']],
    textposition='outside'
))

fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='#888780',
    line_width=1
)

fig.update_layout(
    title=f'Distance from 200-Day MA — '
          f'Top 30 & Bottom 30 '
          f'(green = holding, red = avoiding)',
    yaxis_title='% Above/Below 200d MA',
    xaxis_title='Stock',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_yaxes(ticksuffix='%')
fig.show()

# Histogram — distribution of all distances
fig2 = go.Figure(go.Histogram(
    x=summary['Distance %'],
    nbinsx=50,
    marker_color='#1D9E75',
    opacity=0.8
))
fig2.add_vline(
    x=0,
    line_dash='dash',
    line_color='#D85A30',
    annotation_text='200d MA line'
)
fig2.update_layout(
    title='Distribution of All 500 Stocks '
          'Relative to Their 200d MA',
    xaxis_title='% Above/Below 200d MA',
    yaxis_title='Number of stocks',
    height=400,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.update_xaxes(ticksuffix='%')
fig2.show()

print(f"\n  MARKET HEALTH SUMMARY")
print(f"  {'─'*45}")
print(f"  Stocks above 200d MA: "
      f"{len(above)} ({len(above)/len(summary):.0%})")
print(f"  Stocks below 200d MA: "
      f"{len(below)} ({len(below)/len(summary):.0%})")
print(f"  Avg distance (above): "
      f"+{above['Distance %'].mean():.1f}%")
print(f"  Avg distance (below): "
      f"{below['Distance %'].mean():.1f}%")
print(f"  Strongest trend:      "
      f"{summary.index[0]} "
      f"({summary['Distance %'].iloc[0]:+.1f}%)")
print(f"  Weakest:              "
      f"{summary.index[-1]} "
      f"({summary['Distance %'].iloc[-1]:+.1f}%)")

  200-DAY MA RANKINGS — June 17, 2026

  7 stocks ABOVE 200d MA  |  3 stocks BELOW 200d MA

  Rank   Ticker      Price  200d MA   Distance Status      
  ------ -------- -------- -------- ---------- ------------
  1      XLK      $ 189.02 $ 148.80     +27.0% ABOVE ▲      ◄ HOLDING
  2      QQQ      $ 733.51 $ 626.93     +17.0% ABOVE ▲      ◄ HOLDING
  3      IWM      $ 295.25 $ 256.49     +15.1% ABOVE ▲      ◄ HOLDING
  4      SPY      $ 750.59 $ 685.73      +9.5% ABOVE ▲      ◄ HOLDING
  5      XLE      $  54.97 $  50.48      +8.9% ABOVE ▲      ◄ HOLDING
  6      SLV      $  63.92 $  61.66      +3.7% ABOVE ▲      ◄ HOLDING
  7      XLV      $ 152.32 $ 148.60      +2.5% ABOVE ▲      ◄ HOLDING
  8      TLT      $  86.38 $  86.38      -0.0% BELOW ▼     
  9      IEF      $  94.51 $  94.57      -0.1% BELOW ▼     
  10     GLD      $ 400.37 $ 407.86      -1.8% BELOW ▼     



  MARKET HEALTH SUMMARY
  ─────────────────────────────────────────────
  Stocks above 200d MA: 7 (70%)
  Stocks below 200d MA: 3 (30%)
  Avg distance (above): +11.9%
  Avg distance (below): -0.6%
  Strongest trend:      XLK (+27.0%)
  Weakest:              GLD (-1.8%)


In [15]:
# smart rebalance (only trade what changed)
capital = float(client.get_account().cash) * 0.95 + \
          sum([float(p.market_value)
               for p in client.get_all_positions()])

print(f"Total portfolio value: ${capital:,.2f}")

# currently holdings
current_positions = {p.symbol: float(p.market_value)
                    for p in client.get_all_positions()}

# model holdings
target = {ticker: capital * weight
          for ticker, weight in picks.items()}

print("\n=== CURRENT vs TARGET ===")
all_tickers = set(current_positions.keys()) | set(target.keys())
for ticker in all_tickers:
    current = current_positions.get(ticker, 0)
    wanted  = target.get(ticker, 0)
    diff    = wanted - current
    print(f"  {ticker}: have ${current:,.0f} → want ${wanted:,.0f} "
          f"({'buy' if diff > 0 else 'SELL'} ${abs(diff):,.0f})")

print("\n=== PLACING ORDERS ===")
for ticker in all_tickers:
    current = current_positions.get(ticker, 0)
    wanted  = target.get(ticker, 0)
    diff    = wanted - current

    # Only trade if difference is more than $500
    if abs(diff) < 500:
        print(f"  {ticker}: skipping (difference too small)")
        continue

    if wanted == 0 and current > 0:
        # close the position
        client.close_position(ticker)
        print(f"  SOLD {ticker} (no longer in signal)")

    elif diff > 0:
        # buy more
        order = MarketOrderRequest(
            symbol=ticker,
            notional=round(diff, 2),
            side=OrderSide.BUY,
            time_in_force=TimeInForce.DAY
        )
        client.submit_order(order)
        print(f"  Bought ${diff:,.0f} more of {ticker}")

    elif diff < -500:
        # trim position
        order = MarketOrderRequest(
            symbol=ticker,
            notional=round(abs(diff), 2),
            side=OrderSide.SELL,
            time_in_force=TimeInForce.DAY
        )
        client.submit_order(order)
        print(f"  Trimmed ${abs(diff):,.0f} of {ticker}")

Total portfolio value: $98,585.65

=== CURRENT vs TARGET ===
  WBD: have $10,201 → want $0 (SELL $10,201)
  WMB: have $8,694 → want $0 (SELL $8,694)
  XLV: have $0 → want $14,084 (buy $14,084)
  SLV: have $0 → want $14,084 (buy $14,084)
  CVX: have $5,436 → want $0 (SELL $5,436)
  IWM: have $0 → want $14,084 (buy $14,084)
  DVN: have $10,595 → want $0 (SELL $10,595)
  VLO: have $5,621 → want $0 (SELL $5,621)
  TRGP: have $7,149 → want $0 (SELL $7,149)
  COP: have $5,171 → want $0 (SELL $5,171)
  XOM: have $5,306 → want $0 (SELL $5,306)
  LMT: have $6,019 → want $0 (SELL $6,019)
  BKR: have $5,689 → want $0 (SELL $5,689)
  XLE: have $0 → want $14,084 (buy $14,084)
  TPL: have $4,718 → want $0 (SELL $4,718)
  BG: have $5,862 → want $0 (SELL $5,862)
  SPY: have $0 → want $14,084 (buy $14,084)
  XLK: have $0 → want $14,084 (buy $14,084)
  APA: have $5,182 → want $0 (SELL $5,182)
  QQQ: have $0 → want $14,084 (buy $14,084)
  KMI: have $7,914 → want $0 (SELL $7,914)

=== PLACING ORDERS ===
 

In [16]:
# Chart 1 — Portfolio performance vs buy & hold
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1,
                    subplot_titles=('Portfolio vs Buy & Hold',
                                   'Drawdown',
                                   'Monthly Returns Heatmap'),
                    row_heights=[0.5, 0.25, 0.25],
                    vertical_spacing=0.08)

# Cumulative returns
fig.add_trace(go.Scatter(
    x=cumulative.index,
    y=(cumulative - 1) * 100,
    name='Strategy',
    line=dict(color='#1D9E75', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=buy_hold.index,
    y=(buy_hold - 1) * 100,
    name='Buy & Hold SPY',
    line=dict(color='#888780', width=1.5, dash='dash')
), row=1, col=1)

# Drawdown
dd = (cumulative / cumulative.cummax() - 1) * 100
fig.add_trace(go.Scatter(
    x=dd.index,
    y=dd,
    name='Drawdown',
    fill='tozeroy',
    line=dict(color='#D85A30', width=1),
    fillcolor='rgba(216,90,48,0.15)'
), row=2, col=1)

# Monthly returns bar chart
monthly_rets = port_returns.resample('ME').apply(
    lambda x: (1 + x).prod() - 1) * 100
colors = ['#1D9E75' if r >= 0 else '#D85A30'
          for r in monthly_rets]
fig.add_trace(go.Bar(
    x=monthly_rets.index,
    y=monthly_rets,
    name='Monthly Return',
    marker_color=colors
), row=3, col=1)

fig.update_layout(
    height=800,
    title='200-Day MA Trend Following Strategy',
    showlegend=True,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.update_yaxes(ticksuffix='%')
fig.show()

In [17]:
# Chart 2 — Current signals (which ETFs are above/below 200d MA)
fig2 = go.Figure()

for ticker in tickers:
    fig2.add_trace(go.Scatter(
        x=data.index[-252:],
        y=data[ticker][-252:],
        name=ticker,
        mode='lines',
        visible='legendonly'
    ))

fig2.update_layout(
    title='Price vs 200-Day MA (last 12 months) — click tickers to show',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()


In [18]:
# Chart 3 — Signal heatmap (1 = holding, 0 = in cash for each ETF)
import plotly.express as px

signal_monthly = signal.resample('ME').last()
fig3 = px.imshow(
    signal_monthly.T,
    color_continuous_scale=['#D85A30', '#1D9E75'],
    title='Holdings Heatmap — green = holding, red = in cash',
    aspect='auto'
)
fig3.update_layout(height=400)
fig3.show()

In [28]:
# Backtest Cell
import numpy as np

def sharpe(r, rf=0.02):
    e = r - rf/252
    return float((e.mean() / e.std()) * np.sqrt(252))

def sortino(r, rf=0.02):
    e        = r - rf/252
    neg      = e[e < 0]
    downside = np.sqrt((neg**2).mean()) * np.sqrt(252)
    return float(e.mean() * 252 / downside)

def max_drawdown(r):
    c = (1 + r).cumprod()
    return float(((c - c.cummax()) / c.cummax()).min())

def cagr(r):
    c = (1 + r).cumprod()
    y = len(r) / 252
    return float(c.iloc[-1] ** (1/y) - 1)

def win_rate(r):
    return float((r > 0).sum() / len(r))

spy     = yf.download('SPY', start='2018-01-01',
                       auto_adjust=True,
                       progress=False)['Close'].squeeze()
spy_ret = spy.pct_change(fill_method=None).squeeze()


print("=" * 40)
print("FINAL BACKTEST RESULTS")
print("=" * 40)
print(f"Strategy CAGR:    {cagr(port_returns):.1%}")
print(f"SPY CAGR:         {cagr(spy_ret):.1%}")
print(f"Sharpe Ratio:     {sharpe(port_returns):.2f}")
print(f"Sortino Ratio:    {sortino(port_returns):.2f}")
print(f"Max Drawdown:     {max_drawdown(port_returns):.1%}")
print(f"Win Rate:         {win_rate(port_returns):.1%}")
print(f"SPY Correlation:  "
      f"{port_returns.corr(spy_ret):.2f}")

FINAL BACKTEST RESULTS
Strategy CAGR:    13.0%
SPY CAGR:         14.7%
Sharpe Ratio:     0.76
Sortino Ratio:    0.76
Max Drawdown:     -23.7%
Win Rate:         48.8%
SPY Correlation:  0.47


In [20]:
# Walkforward Cell
def walk_forward_sharpe(returns, split=0.70):
    split_idx   = int(len(returns) * split)
    in_sample   = returns[:split_idx]
    out_sample  = returns[split_idx:]

    is_sharpe   = float(
        (in_sample.mean() /
         in_sample.std()) * np.sqrt(252)
    )
    oos_sharpe  = float(
        (out_sample.mean() /
         out_sample.std()) * np.sqrt(252)
    )
    degradation = ((is_sharpe - oos_sharpe) /
                    abs(is_sharpe)) * 100

    print(f"In-sample Sharpe:      {is_sharpe:.2f}")
    print(f"Out-of-sample Sharpe:  {oos_sharpe:.2f}")
    print(f"Degradation:           {degradation:.1f}%")
    return is_sharpe, oos_sharpe

walk_forward_sharpe(port_returns)

In-sample Sharpe:      0.69
Out-of-sample Sharpe:  1.45
Degradation:           -111.1%


(0.6882355590133662, 1.4529401795752372)

In [23]:
# Kelly Criterion
def kelly_fraction(returns):
    wr    = float((returns > 0).mean())
    avg_w = float(returns[returns > 0].mean())
    avg_l = float(abs(returns[returns < 0].mean()))
    f     = (wr / avg_l) - ((1 - wr) / avg_w)
    return max(0, min(f / 2, 0.25))

kelly = kelly_fraction(port_returns)
print(f"Kelly fraction: {kelly:.1%}")

Kelly fraction: 0.0%
